# Scenario 3: Customer Segmentation Query

### Prompt:

I have three tables called user_activity, transactions, and user_preferences. I need a PostgreSQL query that finds active users who have logged in within the last 30 days, filters those users down to the top 20% based on total spending, and then shows preference trends for those customers. Can you write the query and explain how you handled the date filtering, percentile calculation, and joins?

### Generated Code:


#### Assumed table structure

The prompt only gives the table names, so the query assumes these columns. Rename them to match your schema.

| Table | Columns used |
|---|---|
| `user_activity` | `user_id`, `activity_type` (e.g. `'login'`), `activity_timestamp` (`timestamptz`) |
| `transactions` | `user_id`, `amount`, `status` (e.g. `'completed'`), `transaction_date` |
| `user_preferences` | `user_id`, `preference_category`, `preference_value`, `updated_at` |

#### Query

```sql
WITH active_users AS (
    -- 1. Users with at least one login in the last 30 days (one row per user)
    SELECT DISTINCT user_id
    FROM user_activity
    WHERE activity_type = 'login'
      AND activity_timestamp >= NOW() - INTERVAL '30 days'
      AND activity_timestamp <= NOW()                  -- ignore future-dated rows
),
user_spend AS (
    -- 2. Total completed spend per active user, summed BEFORE any other join
    SELECT a.user_id,
           COALESCE(SUM(t.amount), 0) AS total_spend  -- active users with no purchases count as 0
    FROM active_users a
    LEFT JOIN transactions t
           ON t.user_id = a.user_id
          AND t.status = 'completed'                   -- leave out refunds, failed and pending payments
    GROUP BY a.user_id
),
spend_cutoff AS (
    -- 3. Spend at the 80th percentile of active users
    SELECT PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY total_spend) AS p80
    FROM user_spend
),
top_spenders AS (
    -- 4. Top 20%: at or above the cutoff, and must have spent something
    SELECT s.user_id, s.total_spend
    FROM user_spend s
    CROSS JOIN spend_cutoff c
    WHERE s.total_spend >= c.p80
      AND s.total_spend > 0
),
group_sizes AS (
    SELECT (SELECT COUNT(*) FROM top_spenders)                                   AS n_top,
           (SELECT COUNT(*) FROM active_users) - (SELECT COUNT(*) FROM top_spenders) AS n_other
),
pref_counts AS (
    -- 5. How many top spenders vs. other active users hold each preference
    SELECT p.preference_category,
           p.preference_value,
           COUNT(DISTINCT p.user_id) FILTER (WHERE ts.user_id IS NOT NULL) AS top_users,
           COUNT(DISTINCT p.user_id) FILTER (WHERE ts.user_id IS NULL)     AS other_users
    FROM user_preferences p
    JOIN active_users a        ON a.user_id  = p.user_id   -- active users only
    LEFT JOIN top_spenders ts  ON ts.user_id = p.user_id   -- tag which ones are top spenders
    GROUP BY p.preference_category, p.preference_value
)
-- 6. Preference trends for top spenders, compared with the rest of the active users
SELECT pc.preference_category,
       pc.preference_value,
       pc.top_users,
       ROUND(100.0 * pc.top_users   / NULLIF(g.n_top, 0),   1) AS pct_of_top_spenders,
       ROUND(100.0 * pc.other_users / NULLIF(g.n_other, 0), 1) AS pct_of_other_active,
       ROUND( (pc.top_users::numeric   / NULLIF(g.n_top, 0))
            / NULLIF(pc.other_users::numeric / NULLIF(g.n_other, 0), 0), 2) AS lift
FROM pref_counts pc
CROSS JOIN group_sizes g
WHERE pc.top_users > 0
ORDER BY pc.preference_category, pct_of_top_spenders DESC;
```

Each output row is one preference and shows the share of top spenders who have it next to the share of other active users. **`lift`** is the ratio of the two:
- **Above 1:** the preference is more common among top spenders. For example, 1.8 means 80% more common.
- **Below 1:** it is less common among them.
- **Blank:** no other active user has that preference, so there's nothing to compare against.

#### How the date filtering works

- **Rolling 30 days:** `activity_timestamp >= NOW() - INTERVAL '30 days'` means the last 30 × 24 hours up to the moment the query runs. For whole calendar days instead (midnight 30 days ago through today), use `>= CURRENT_DATE - 30`.
- **The column is compared directly.** It is not wrapped in a function such as `DATE(activity_timestamp)`, so PostgreSQL can use an index on `(activity_type, activity_timestamp)` instead of scanning the whole table.
- **Future-dated rows are excluded** (`<= NOW()`), so a bad clock or test data can't make someone look active.
- **Time zones:** if the column is `timestamptz`, `NOW()` compares correctly whatever the server's time zone. If it's a plain `timestamp`, PostgreSQL uses the session's time zone setting, so the 30-day cutoff can shift by a few hours unless the stored times and the session use the same zone.
- **Spending covers all completed transactions**, because the prompt says "total spending". To rank only by recent spending, add `AND t.transaction_date >= NOW() - INTERVAL '30 days'` to the transactions join condition. It must go in the `ON` clause, not in `WHERE`. Putting it in `WHERE` would silently drop users with no recent purchases.

#### How the percentile works

- **`PERCENTILE_CONT(0.8)`** finds the spending amount that 80% of active users are at or below. Anyone at or above that amount is in the top 20%.
- **The 20% is of active users only**, because the percentile is calculated from `user_spend`, which already contains only active users.
- **Active users with no purchases count as 0 spend** (the `LEFT JOIN` plus `COALESCE`). Without that, they would be left out and the cutoff would be measured against buyers only.
- **Ties are treated the same way:** everyone with exactly the cutoff amount is included. The group can therefore come out slightly larger than 20% when many users tie at the cutoff. `NTILE(5)` would split tied users between groups arbitrarily, which is worse.
- **`total_spend > 0`:** if more than 80% of active users spent nothing, the cutoff would be 0 and every non-buyer would count as a "top spender". This condition prevents that.

#### How the joins work

- **Activity → distinct users.** A user may log in dozens of times, so `DISTINCT user_id` reduces the logins to one row per user before anything else is joined.
- **Transactions are summed before joining anything else.** Joining login rows to transaction rows directly would multiply them: 10 logins × 5 purchases = 50 rows, and total spend would be counted 10 times over. Aggregating per user in `user_spend` avoids this.
- **`LEFT JOIN` to transactions** keeps active users who haven't bought anything (see above). The `status = 'completed'` filter sits in the `ON` clause for the same reason.
- **Preferences are joined last.** They join to active users, with a `LEFT JOIN` to top spenders so each user is tagged as top or other. A user can have several preference rows, so `COUNT(DISTINCT user_id)` counts each person only once per preference.
- **`NULLIF(..., 0)`** prevents division-by-zero errors when a group is empty.

#### Optional: preference trends over time

To see how top spenders' preferences change month by month, keep the same `WITH` block and replace the final `SELECT` with:

```sql
SELECT DATE_TRUNC('month', p.updated_at) AS month,
       p.preference_category,
       p.preference_value,
       COUNT(DISTINCT p.user_id)          AS top_spenders_with_preference
FROM user_preferences p
JOIN top_spenders ts ON ts.user_id = p.user_id
WHERE p.updated_at >= DATE_TRUNC('month', NOW()) - INTERVAL '6 months'
GROUP BY 1, 2, 3
ORDER BY month, p.preference_category, top_spenders_with_preference DESC;
```

This counts preferences by the month they were set or last updated. It only shows true history if `user_preferences` keeps old rows. If it overwrites each preference in place, only the latest value per user is available.

### Follow-up Prompts:

You used some columns that aren't actually in the schema. Can you redo the query using only the columns I was given?

user_activity: user_id, last_login_date, feature_usage_count, account_type

transactions: transaction_id, user_id, transaction_date, amount, platform

user_preferences: user_id, communication_preference, interface_theme, notification_settings

I still need it to find users who logged in within the last 30 days, get the top 20% based on total spending, and show the preference trends for those users. Also explain how you're doing the top 20% calculation and see if the query can be simplified at all.

### Final Solution:

#### Query (uses only the given columns)

```sql
WITH active_spend AS (
    -- Active users (logged in within the last 30 days) and their total spend
    SELECT a.user_id,
           COALESCE(SUM(t.amount), 0) AS total_spend   -- active users with no purchases count as 0
    FROM user_activity a
    LEFT JOIN transactions t ON t.user_id = a.user_id
    WHERE a.last_login_date >= CURRENT_DATE - INTERVAL '30 days'
    GROUP BY a.user_id
),
top_spenders AS (
    -- Top 20% of active users by total spend
    SELECT user_id
    FROM (
        SELECT user_id,
               total_spend,
               CUME_DIST() OVER (ORDER BY total_spend DESC) AS spend_share
        FROM active_spend
    ) ranked
    WHERE spend_share <= 0.20
      AND total_spend > 0
)
-- Preference trends: how common each setting value is among top spenders
SELECT pref.setting,
       pref.value,
       COUNT(*) AS top_spenders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY pref.setting), 1) AS pct_of_top_spenders
FROM top_spenders ts
JOIN user_preferences p ON p.user_id = ts.user_id
CROSS JOIN LATERAL (VALUES
    ('communication_preference', COALESCE(p.communication_preference::text, 'not set')),
    ('interface_theme',          COALESCE(p.interface_theme::text,          'not set')),
    ('notification_settings',    COALESCE(p.notification_settings::text,    'not set'))
) AS pref(setting, value)
GROUP BY pref.setting, pref.value
ORDER BY pref.setting, top_spenders DESC;
```

**Example output shape:**

| setting | value | top_spenders | pct_of_top_spenders |
|---|---|---|---|
| communication_preference | email | 120 | 60.0 |
| communication_preference | sms | 80 | 40.0 |
| interface_theme | dark | 150 | 75.0 |
| … | … | … | … |

#### How the top 20% is calculated

1. **`active_spend`** gives one row per active user with their total spend. Active users who never bought anything get 0 instead of being left out, so the 20% is measured against *all* active users, not just buyers.
2. **`CUME_DIST() OVER (ORDER BY total_spend DESC)`** ranks users from highest to lowest spend. For each user it returns the share of active users who spent **at least as much** as they did:
   - the biggest spender out of 100 users gets 0.01
   - the 20th biggest gets 0.20
   - the 21st gets 0.21
3. **`spend_share <= 0.20`** keeps users in the top fifth.
4. **`total_spend > 0`** makes sure a user with no spending is never counted as a "top spender", which could happen when most active users bought nothing.

**Ties:** users with the same total get the same `spend_share`, so tied users are always kept or dropped together. If a tie falls exactly on the 20% line, the whole tied group is left out, so the result can be slightly *under* 20%. That's usually better than `NTILE(5)`, which would split tied users between groups arbitrarily. If you'd rather include the whole tied group (and come out slightly *over* 20%), use `PERCENT_RANK() OVER (ORDER BY total_spend DESC) < 0.20` instead.

#### Date filtering

- **`last_login_date >= CURRENT_DATE - INTERVAL '30 days'`** keeps users whose most recent login falls on or after the date 30 days ago.
- **No `DISTINCT` is needed:** `user_activity` stores each user's *last* login date, which suggests one row per user.
- **Spending covers all transactions,** because the prompt asks for total spending. To rank by recent spending only, add `AND t.transaction_date >= CURRENT_DATE - INTERVAL '30 days'` to the `LEFT JOIN ... ON` line.

#### Preference trends

`user_preferences` has three settings columns and no date, so the "trend" here is how the top spenders are distributed across each setting's values. There's no change over time to show.
- **`CROSS JOIN LATERAL (VALUES ...)`** turns the three columns into rows (setting, value), so one `GROUP BY` covers all three instead of needing three separate queries joined with `UNION`.
- **`::text`** lets the three columns be combined even if they have different types (for example, if `notification_settings` is a boolean or JSON).
- **Missing values** show up as `'not set'`.
- **`pct_of_top_spenders`** is each value's share within its setting, so the percentages for each setting add up to 100.

#### How this is simpler than the Generated Code version

| Before | Now |
|---|---|
| 6 CTEs | 2 CTEs plus the final `SELECT` |
| `DISTINCT` on login events | not needed: one row per user with `last_login_date` |
| A separate `spend_cutoff` CTE with `PERCENTILE_CONT` plus a `CROSS JOIN` | one `CUME_DIST()` window function |
| A `group_sizes` CTE, `NULLIF` and a lift calculation | removed; percentages come from one window `SUM` |
| `status = 'completed'` filter | removed: there's no `status` column |

**Assumptions to check:**
- **One row per user in `user_activity` and in `user_preferences`.** If either has more, spend or preference counts would be multiplied. The fix is to use `SELECT DISTINCT user_id` for the active users and `COUNT(DISTINCT ts.user_id)` for the preference counts.
- **Refunds:** there's no `status` column, so refunds (if they're stored as negative amounts) are subtracted from the user's total.

**Optional comparison with other active users:** add a column to `top_spenders` marking whether each user is a top spender, keep every active user instead of filtering, and add that column to the `GROUP BY` and the `PARTITION BY`.